# Phát hiện và tự động sửa lỗi chính tả tiếng Việt với ViT5-base

Notebook nghiên cứu gồm 21 mục. Tải notebook và **vietspell-colab-source.zip** lên **Drive của tôi → Colab Notebooks → vietnamese-spell-checker**. Mở notebook trong Colab, chọn **Runtime → Change runtime type → GPU** (T4 nếu có), rồi chạy tuần tự. Mặc định chạy hai cấu hình ViT5 và một baseline trên cùng phân chia VSEC. ByT5 là đối chứng tùy chọn.

Kết quả được tính từ dữ liệu, log và dự đoán do chính lượt chạy tạo ra. Triển khai giao diện nằm ngoài phạm vi notebook.

## 01. Cài đặt môi trường

PyTorch thực hiện tính toán tensor trên GPU và Transformers cung cấp tokenizer, mô hình Seq2Seq và Trainer. Tên GPU và phiên bản thư viện được đọc từ phiên Colab.

In [ ]:
from google.colab import drive
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import os
import subprocess
import sys

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/Colab Notebooks/vietnamese-spell-checker")
ROOT = Path("/content/vietnamese-spell-checker")
SOURCE = DRIVE / "vietspell-colab-source.zip"
if not SOURCE.is_file():
    raise FileNotFoundError(f"Tải vietspell-colab-source.zip vào {DRIVE}")
ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(SOURCE) as archive:
    archive.extractall(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def run(*arguments):
    subprocess.run([sys.executable, "-X", "utf8", "-u", *map(str, arguments)], cwd=ROOT, check=True)

run("-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "matplotlib")
import torch
import transformers
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn GPU trong Runtime rồi chạy lại ô thiết lập.")
subprocess.run(["nvidia-smi"], check=True)
display(pd.DataFrame([{
    "Python": sys.version.split()[0], "PyTorch": torch.__version__,
    "Transformers": transformers.__version__, "CUDA": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0),
    "VRAM (GiB)": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
}]))

## 02. Cấu hình huấn luyện

Seed cố định cách chia dữ liệu và cách sinh nhiễu. Batch hiệu dụng trên một GPU bằng batch mỗi bước nhân số bước tích lũy gradient. Hai cấu hình ViT5 giữ batch hiệu dụng giống nhau. Chênh lệch giữa chúng đánh giá cả cấu hình, vì nhiều tham số thay đổi cùng lúc.

In [ ]:
import json
import random
import numpy as np
from src.utils.helpers import load_config, set_seed

SEED = 42
NOISE_SIZE, IDENTITY_SIZE, NOISE_PROB = 5000, 1868, .20
RUN_TUNING = True
RUN_BYT5 = False
KEEP_RECOVERY_ON_DRIVE = False
RUN_ID = "vsec-study-01"  # Đổi tên khi muốn thực nghiệm mới
VIT5_CONFIG = "configs/config.yaml"
BYT5_CONFIG = "configs/byt5.yaml"
base_config = load_config(VIT5_CONFIG)
assert base_config["seed"] == load_config(BYT5_CONFIG)["seed"] == SEED
set_seed(SEED)

RUNS = {
    "vit5-base": dict(model="vit5", config=VIT5_CONFIG, lr=1e-4, epochs=3, batch=2, accumulation=16),
    "vit5-tuned": dict(model="vit5", config=VIT5_CONFIG, lr=5e-5, epochs=4, batch=4, accumulation=8),
    "byt5": dict(model="byt5", config=BYT5_CONFIG, lr=1e-4, epochs=3, batch=1, accumulation=16),
}
WORK = ROOT / "experiments" / RUN_ID
RUN_PARENT = DRIVE / "training-runs" / RUN_ID if KEEP_RECOVERY_ON_DRIVE else WORK / "runs"
ARTIFACTS = DRIVE / "training-exports" / RUN_ID
display(pd.DataFrame.from_dict(RUNS, orient="index").assign(
    effective_batch=lambda frame: frame["batch"] * frame["accumulation"]
))
print(f"Seed={SEED}. Tinh chỉnh={RUN_TUNING}. ByT5={RUN_BYT5}. Checkpoint khôi phục: {RUN_PARENT}")

## 03. Thu thập dữ liệu

VSEC Corpus chứa các annotation lỗi chính tả và phương án sửa do người gán nhãn. Notebook tải JSONL từ GitHub, giữ nguyên bản thô và ghi SHA-256 để truy vết. [Nguồn VSEC](https://github.com/VSEC2021/VSEC)

In [ ]:
from urllib.request import urlretrieve
from src.data.prepare_vsec import RAW, URL, file_hash

if not RAW.is_file():
    RAW.parent.mkdir(parents=True, exist_ok=True)
    partial = RAW.with_suffix(".download")
    urlretrieve(URL, partial)
    partial.replace(RAW)
RAW_SHA256 = file_hash(RAW)
display(pd.DataFrame([{"Nguồn": URL, "Tệp": RAW.name, "Byte": RAW.stat().st_size, "SHA-256": RAW_SHA256}]))

## 04. Phân tích dữ liệu (EDA)

Một annotation được tính là lỗi khi is_correct là False. Độ dài được đo bằng số mục annotation. Tần suất cặp sai → đúng dùng đáp án đầu tiên nếu có nhiều phương án. Các thống kê này mô tả dữ liệu, còn việc chọn mô hình về sau chỉ dùng Validation.

In [ ]:
from collections import Counter
from src.data.prepare_vsec import parse_record

eda_records, error_pairs, parsed_rows, rejected_lines = [], Counter(), [], []
with RAW.open(encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, 1):
        try:
            item = json.loads(line)
            pair = parse_record(item, line_number)
        except (ValueError, TypeError, KeyError, IndexError, AttributeError):
            rejected_lines.append(line_number)
            continue
        annotations = item["annotations"]
        errors = [annotation for annotation in annotations if not annotation["is_correct"]]
        eda_records.append({"Lỗi": len(errors), "Độ dài": len(annotations)})
        parsed_rows.append(pair)
        error_pairs.update((annotation["current_syllable"], annotation["alternative_syllables"][0])
                           for annotation in errors)

eda = pd.DataFrame(eda_records)
display(pd.DataFrame([{"Số câu": len(eda), "Số lỗi": eda["Lỗi"].sum(),
                       "Lỗi/câu": eda["Lỗi"].mean(), "Dòng lỗi định dạng": len(rejected_lines)}]).round(2))
display(eda["Độ dài"].describe().to_frame().round(2))
display(pd.DataFrame([(a, b, n) for (a, b), n in error_pairs.most_common(10)],
                     columns=["Sai", "Đáp án thứ nhất", "Tần suất"]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eda["Lỗi"].value_counts().sort_index().plot.bar(ax=axes[0], title="Số lỗi mỗi câu")
axes[0].set_ylabel("Số câu")
eda["Độ dài"].plot.hist(bins=30, ax=axes[1], title="Độ dài câu theo annotation")
axes[1].set_xlabel("Số annotation")
plt.tight_layout()
plt.show()

## 05. Tiền xử lý dữ liệu

Chuẩn Unicode NFC gom các biểu diễn tương đương của chữ tiếng Việt. Câu đầu vào lấy từ annotation gốc, câu đích dùng phương án sửa cho các âm tiết sai. Khi loại cặp trùng, quy trình vẫn giữ quan hệ giữa câu gốc và các bản sao để phân tập theo nhóm.

In [ ]:
from src.data.prepare_vsec import group_rows
from src.data.preprocess import normalize_vietnamese_text

unique_pairs = {(row["input_text"], row["target_text"]) for row in parsed_rows}
exact_groups = group_rows(parsed_rows)
assert all(row["input_text"] == normalize_vietnamese_text(row["input_text"])
           and row["target_text"] == normalize_vietnamese_text(row["target_text"]) for row in parsed_rows)
display(pd.DataFrame([{"Cặp trước lọc": len(parsed_rows), "Cặp duy nhất": len(unique_pairs),
                       "Bản trùng": len(parsed_rows) - len(unique_pairs), "Nhóm nội dung": len(exact_groups)}]))
display(pd.DataFrame(parsed_rows).loc[:, ["id", "input_text", "target_text", "error_count"]].head())

## 06. Phân chia dữ liệu

Mục tiêu là Train 80%, Validation 10%, Test 10%. Nhóm nội dung trùng và gần trùng đi cùng nhau, nên tỷ lệ thực tế có thể chênh nhẹ. Manifest ghi thành viên và SHA-256 của từng tập.

In [ ]:
from src.data.prepare_vsec import MANIFEST, PROCESSED, prepare

manifest = prepare(raw=RAW, output=PROCESSED, manifest_path=MANIFEST, seed=SEED)
assert manifest["duplicates"] == len(parsed_rows) - len(unique_pairs)
split_table = pd.DataFrame([{"Tập": name, "Số câu": item["samples"],
                             "Lỗi gán nhãn": item["annotated_errors"],
                             "Câu giữ nguyên": item["clean_sentences"],
                             "Tỷ lệ (%)": 100 * item["samples"] / len(unique_pairs)}
                            for name, item in manifest["splits"].items()])
display(split_table.round(2))

## 07. Kiểm tra Data Leakage

Đối chiếu ID, khóa nội dung của câu gốc/sai/đúng, chữ ký tệp và các cặp gần trùng theo bộ phát hiện của dự án. Nếu kiểm tra thất bại, dừng trước khi huấn luyện.

In [ ]:
from itertools import combinations
from src.data.prepare_vsec import content_keys
from src.data.validate_split import validate

splits, manifest = validate()
keys = {name: set().union(*(content_keys(row) for row in rows)) for name, rows in splits.items()}
ids = {name: {row["id"] for row in rows} for name, rows in splits.items()}
leaks = pd.DataFrame([{"Hai tập": f"{a} / {b}", "ID trùng": len(ids[a] & ids[b]),
                       "Nội dung trùng": len(keys[a] & keys[b])}
                      for a, b in combinations(splits, 2)])
display(leaks)
assert not leaks[["ID trùng", "Nội dung trùng"]].to_numpy().any()
print("Đã kiểm tra SHA-256, ID và cặp gần trùng theo quy tắc của dự án.")

## 08. Data Augmentation

Chỉ sinh mẫu từ Train. Ba nhóm nhiễu gồm mất toàn bộ dấu, mất dấu một phần và nhiễu hỗn hợp. Nhiễu hỗn hợp có thể đổi phụ âm, gõ nhầm hoặc đảo ký tự. Cặp câu sạch giúp mô hình học giữ nguyên văn bản đúng.

In [ ]:
from generate_noise_data import augment
from src.data.noise_synthesizer import INITIALS, FINALS
from src.postprocessing import WORD, without_accents
from src.training.metrics import edits_for

extra = augment(splits, NOISE_SIZE, NOISE_PROB, SEED, identity_size=IDENTITY_SIZE)
held_out = keys["val"] | keys["test"]
held_out |= {without_accents(key) for key in held_out}
assert all(row["parent_id"] in ids["train"] and
           not ({without_accents(key) for key in content_keys(row)} & held_out) for row in extra)

def has_consonant_change(row):
    clean = WORD.findall(row["target_text"])
    for (index, kind), replacement in edits_for(row["target_text"], row["input_text"]).items():
        if kind != "word" or len(replacement) != 1:
            continue
        a, b = clean[index].casefold(), replacement[0].casefold()
        if any(a.startswith(old) and b == new + a[len(old):] for old, new in INITIALS.items()):
            return True
        if any(a.endswith(old) and b == a[:-len(old)] + new for old, new in FINALS.items()):
            return True
    return False

augmentation = pd.DataFrame(extra)
display(augmentation["noise_type"].value_counts().rename("Số mẫu").to_frame())
examples = [row for row in extra if row["noise_type"] == "mixed" and has_consonant_change(row)]
assert examples, "Cấu hình hiện tại không sinh mẫu đổi phụ âm."
print(f"{len(examples)} mẫu hỗn hợp có đổi phụ âm theo bảng quy tắc.")
display(pd.DataFrame(examples[:5])[["parent_id", "input_text", "target_text"]])

## 09. Tải mô hình pretrained

ViT5-base là điểm khởi đầu cho bài toán Seq2Seq. Mô hình sẽ học ánh xạ từ câu sai sang câu đúng ở phần fine-tuning. Cache Hugging Face được dùng lại cho các lượt chạy. [VietAI/vit5-base](https://huggingface.co/VietAI/vit5-base)

In [ ]:
from src.models.pretrained import VIT5_MODEL
from src.models.spell_corrector import VietnameseSpellCorrector

set_seed(SEED)
pretrained = VietnameseSpellCorrector(VIT5_MODEL, device=torch.device("cpu"))
print("Mô hình:", pretrained.model.config.name_or_path)
print("Revision:", pretrained.model.config._commit_hash)
print("Tokenizer:", type(pretrained.tokenizer).__name__)

## 10. Phân tích kiến trúc

Encoder biểu diễn câu đầu vào và decoder sinh chuỗi đã sửa. Số lớp, kích thước ẩn và số tham số được đọc từ chính mô hình đã tải.

In [ ]:
architecture = pretrained.model.config
display(pd.DataFrame([{
    "Kiến trúc": type(pretrained.model).__name__, "Encoder–Decoder": architecture.is_encoder_decoder,
    "Encoder layers": architecture.num_layers, "Decoder layers": architecture.num_decoder_layers,
    "Attention heads": architecture.num_heads, "Hidden size": architecture.d_model,
    "Vocab size": architecture.vocab_size,
    "Tham số": sum(parameter.numel() for parameter in pretrained.model.parameters()),
    "Tham số có gradient": sum(parameter.numel() for parameter in pretrained.model.parameters() if parameter.requires_grad),
}]))

## 11. Tokenization và trích xuất đặc trưng

SentencePiece chuyển văn bản thành input IDs. Attention mask đánh dấu token hợp lệ và embedding ánh xạ ID sang vectơ. Padding của nhãn có giá trị -100 để được bỏ qua khi tính loss.

In [ ]:
from src.data.dataset import SpellingDataset, build_data_collator

model_settings = base_config["model"]
limits = (model_settings["max_source_length"], model_settings["max_target_length"], model_settings["prefix"])
preview = SpellingDataset(splits["train"][:16], pretrained.tokenizer, *limits)
batch = build_data_collator(pretrained.tokenizer, pretrained.model)([preview[i] for i in range(min(2, len(preview)))])
with torch.inference_mode():
    embeddings = pretrained.model.get_input_embeddings()(batch["input_ids"])
display(pd.DataFrame([{"Tensor": name, "Shape": tuple(values.shape), "Ví dụ dòng đầu": values[0, :16].tolist()}
                      for name, values in batch.items() if name in {"input_ids", "attention_mask", "labels"}]))
print("Embedding shape (batch, sequence, hidden):", tuple(embeddings.shape))
print("Vector minh họa:", embeddings[0, 0, :8].tolist())

## 12. Thiết lập huấn luyện

T5 tính Cross-Entropy giữa logits và token đích, bỏ qua padding nhãn. Trainer dùng AdamW và lịch learning rate tuyến tính với warmup. Kiểm tra một lượt forward và đọc TrainingArguments trước khi chạy huấn luyện thật.

In [ ]:
import gc
from copy import deepcopy
from src.training.trainer import SpellingTrainer

preview_config = deepcopy(base_config)
preview_config["training"]["output_dir"] = str(WORK / "preview")
with torch.inference_mode():
    forward_loss = pretrained.model(**batch).loss.item()
assert np.isfinite(forward_loss)
preview_trainer = SpellingTrainer(pretrained, preview, preview, preview_config, smoke_steps=1)
arguments = preview_trainer.runner.args
display(pd.DataFrame([{"Loss minibatch ban đầu": forward_loss,
                       "Optimizer": arguments.optim.value, "Scheduler": arguments.lr_scheduler_type.value,
                       "Warmup ratio": arguments.warmup_ratio, "Weight decay": arguments.weight_decay,
                       "FP16": arguments.fp16, "BF16": arguments.bf16}]))
del preview_trainer, arguments, pretrained, embeddings, batch
gc.collect()
torch.cuda.empty_cache()

## 13. Fine-tuning ViT5-base

Chạy vòng huấn luyện thật trên GPU với Train đã tăng cường. Validation loss được ghi cuối mỗi epoch. Khi mở lại notebook, có thể tiếp tục từ checkpoint còn trên ổ đĩa. Đặt KEEP_RECOVERY_ON_DRIVE = True trước khi chạy nếu cần khôi phục qua phiên Colab mới và Drive còn đủ dung lượng.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def train_trial(name):
    settings = RUNS[name]
    output = RUN_PARENT / name
    completed = output / "model_manifest.json"
    if not completed.is_file():
        resume = ["--resume"] if output.is_dir() and any(output.iterdir()) else []
        run("train.py", "--config", settings["config"], "--output", output,
            "--epochs", settings["epochs"], "--batch_size", settings["batch"],
            "--gradient-accumulation", settings["accumulation"], "--lr", settings["lr"],
            "--noise-size", NOISE_SIZE, "--noise-prob", NOISE_PROB, "--identity-size", IDENTITY_SIZE, *resume)
    meta = read_json(completed)
    assert meta["completed"] and not meta["smoke"] and meta["global_steps"] > 0
    assert meta["raw_sha256"] == RAW_SHA256 and meta["seed"] == SEED
    assert meta["splits"] == {key: entry["sha256"] for key, entry in manifest["splits"].items()}
    assert meta["noise_size"] == NOISE_SIZE and meta["noise_probability"] == NOISE_PROB
    assert meta["identities"] == IDENTITY_SIZE
    assert meta["training"]["num_train_epochs"] == settings["epochs"]
    assert meta["training"]["learning_rate"] == settings["lr"]
    assert meta["training"]["per_device_train_batch_size"] == settings["batch"]
    assert meta["training"]["gradient_accumulation_steps"] == settings["accumulation"]
    print(name, "→", meta["global_steps"], "bước cập nhật")
    return output

runs = {"vit5-base": train_trial("vit5-base")}

## 14. Trực quan hóa huấn luyện

Biểu đồ dùng lịch sử Trainer. Training loss được ghi theo bước và Validation loss được ghi cuối epoch. Biểu đồ riêng cho learning rate cho thấy warmup và quá trình giảm.

In [ ]:
def plot_history(output, name):
    history = pd.DataFrame(read_json(output / "training_history.json"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for metric, label in (("loss", "Training Loss"), ("eval_loss", "Validation Loss")):
        series = history.dropna(subset=[metric])
        axes[0].plot(series["step"], series[metric], marker="." if metric == "eval_loss" else None, label=label)
    axes[0].set(xlabel="Bước cập nhật", ylabel="Loss")
    axes[0].legend()
    learning = history.dropna(subset=["learning_rate"])
    axes[1].plot(learning["step"], learning["learning_rate"])
    axes[1].set(xlabel="Bước cập nhật", ylabel="Learning rate")
    fig.suptitle(name)
    fig.tight_layout()
    plt.show()

plot_history(runs["vit5-base"], "ViT5-base")

## 15. Tối ưu hóa mô hình

Thử cấu hình thứ hai với learning rate, số epoch và batch size khác. Chọn checkpoint và beam từ Validation Correction F1, ưu tiên tỷ lệ giữ câu sạch nếu bằng điểm. Test chưa được dùng để chọn.

In [ ]:
if RUN_TUNING:
    runs["vit5-tuned"] = train_trial("vit5-tuned")
    plot_history(runs["vit5-tuned"], "ViT5 · cấu hình thứ hai")

def select_model(model, candidates):
    selected = WORK / "selected" / model / "best_model"
    candidate_paths = [path / "best_model" for path in candidates]
    candidate_ids = {read_json(path / "training_meta.json")["training_id"] for path in candidate_paths}
    if not selected.is_dir():
        run("select_checkpoint.py", "--model", model, "--checkpoints", *candidate_paths,
            "--beams", 1, 2, "--output", selected)
    meta = read_json(selected / "training_meta.json")
    calibrated_ids = {row["training_id"] for row in meta["calibration"]["candidates"]}
    assert candidate_ids == calibrated_ids, "Ứng viên đã thay đổi. Hãy dùng RUN_ID mới."
    return selected, meta

vit5_path, vit5_meta = select_model("vit5", list(runs.values()))
candidate_names = {read_json(path / "model_manifest.json")["training_id"]: name for name, path in runs.items()}
calibration = pd.DataFrame([{"Cấu hình": candidate_names[row["training_id"]], "Beam": row["beams"],
                             "Ngưỡng log-score": row["min_log_probability"],
                             "Correction F1": row["correction"]["f1"],
                             "Câu sạch bị đổi (%)": 100 * row["clean_modified_rate"]}
                            for row in vit5_meta["calibration"]["candidates"]])
display(calibration.sort_values(["Correction F1", "Câu sạch bị đổi (%)"], ascending=[False, True]))
print("Cấu hình ViT5 được chọn:", candidate_names[vit5_meta["training_id"]])

## 16. Đánh giá mô hình

Đo Exact Match ở cấp câu và Accuracy theo token căn chỉnh. Detection tính đúng vị trí lỗi, còn Correction yêu cầu đúng cả từ thay thế. BLEU, CER và WER đo mức tương đồng bề mặt. Các chỉ số dưới đây đến từ Test độc lập.

In [ ]:
RESULTS = ROOT / "experiments" / "results"

def evaluate_trial(model, name, checkpoint=None, split="test"):
    label = f"{RUN_ID}-{name}"
    command = ["evaluate.py", "--model", model, "--split", split, "--run_name", label]
    if checkpoint:
        command += ["--model_path", checkpoint]
    run(*command)
    result = read_json(RESULTS / split / f"{label}.json")
    if split == "test":
        assert result["split_sha256"] == manifest["splits"]["test"]["sha256"]
    if checkpoint:
        assert result["training_id"] == read_json(checkpoint / "training_meta.json")["training_id"]
    return result

def metric_row(name, report):
    m = report["metrics"]
    return {"Mô hình": name, "Số câu": m["num_samples"], "Exact Match (%)": 100 * m["exact_match"],
            "Token Accuracy (%)": 100 * m["token_accuracy"],
            "Detection Precision (%)": m["detection"]["precision"],
            "Detection Recall (%)": m["detection"]["recall"], "Detection F1": m["detection"]["f1"],
            "Correction Precision (%)": m["correction"]["precision"],
            "Correction Recall (%)": m["correction"]["recall"], "Correction F1": m["correction"]["f1"],
            "BLEU": m["bleu"], "CER (%)": 100 * m["cer"], "WER (%)": 100 * m["wer"]}

reports = {"ViT5": evaluate_trial("vit5", "vit5-test", vit5_path)}
display(pd.DataFrame([metric_row("ViT5", reports["ViT5"])]).round(2))
clean_report = evaluate_trial("vit5", "vit5-clean", vit5_path, split="clean_test")
print("Tỷ lệ giữ nguyên câu sạch:", f"{clean_report['metrics']['exact_match']:.2%}")

## 17. So sánh thực nghiệm

Baseline N-gram + Levenshtein xây dựng từ Train và ngữ liệu bổ trợ đã lọc trùng với VSEC. ByT5 chỉ được ghi điểm khi bật và hoàn tất huấn luyện. Mọi mô hình được chấm trên cùng tệp Test và cùng bộ chỉ số.

In [ ]:
reports["N-gram + Levenshtein"] = evaluate_trial("baseline", "baseline-test")
selected_models = {"vit5": vit5_path}
if RUN_BYT5:
    runs["byt5"] = train_trial("byt5")
    byt5_path, _ = select_model("byt5", [runs["byt5"]])
    reports["ByT5"] = evaluate_trial("byt5", "byt5-test", byt5_path)
    selected_models["byt5"] = byt5_path
else:
    print("ByT5: chưa chạy trong thực nghiệm này.")

comparison = pd.DataFrame([metric_row(name, report) for name, report in reports.items()])
display(comparison.round(2))
ax = comparison.set_index("Mô hình")[["Detection F1", "Correction F1", "Exact Match (%)"]].plot.bar(rot=0, ylim=(0, 100))
ax.set_ylabel("Điểm (%)")
plt.tight_layout()
plt.show()

## 18. Phân tích lỗi

Bỏ sót là vị trí lỗi gốc không được sửa. Sửa sai là đúng vị trí nhưng đề xuất sai. Sửa quá mức là chỉnh ở vị trí không có lỗi. Một câu có thể thuộc nhiều nhóm.

In [ ]:
from src.data.prepare_vsec import read_rows

prediction_file = RESULTS / "test" / f"{RUN_ID}-vit5-test.predictions.jsonl"
predictions = pd.DataFrame(read_rows(prediction_file))
failure_types = pd.DataFrame([row or {} for row in predictions["errors"]]).reindex(
    columns=["missed", "wrong_replacement", "overcorrected"], fill_value=0
).fillna(0).astype(int)
error_names = {"missed": "Bỏ sót lỗi", "wrong_replacement": "Sửa sai", "overcorrected": "Sửa quá mức"}
error_totals = pd.DataFrame({"Vị trí": failure_types.sum(), "Câu có lỗi": failure_types.gt(0).sum()}).rename(index=error_names)
display(error_totals)
error_totals["Vị trí"].plot.bar(rot=0, title="Nhóm lỗi trên VSEC Test")
plt.tight_layout()
plt.show()
for key, name in error_names.items():
    print(name)
    display(predictions.loc[failure_types[key] > 0,
                            ["input_text", "matched_reference", "prediction"]].head(3))

## 19. Kiểm thử mô hình với câu mới

Chạy checkpoint đã chọn trên câu tự soạn gồm mất dấu, lỗi phụ âm, lỗi gõ phím, lỗi ngữ cảnh và câu sạch. Các đáp án minh họa do người viết cung cấp, còn dự đoán được sinh trực tiếp. Kết quả minh họa được trình bày riêng với điểm VSEC Test.

In [ ]:
from src.inference import SpellInferencePipeline
from src.data.prepare_vsec import text_key

new_cases = [
    ("Mất dấu", "sang nay doi truc den som de kiem tra duong truyen.",
     "sáng nay đội trực đến sớm để kiểm tra đường truyền."),
    ("Phụ âm đầu", "Tổ kỹ thuật đang chực ở phòng thu.", "Tổ kỹ thuật đang trực ở phòng thu."),
    ("Gõ thừa và sai dấu", "Cô kiểm traa bản vẽ trước khi chuyễn cho cả nhóm.",
     "Cô kiểm tra bản vẽ trước khi chuyển cho cả nhóm."),
    ("Ngữ cảnh", "Bạn nhớ sữa tiêu đề rồi gửi lại cho cô.", "Bạn nhớ sửa tiêu đề rồi gửi lại cho cô."),
    ("Câu sạch", "Em cất lọ thủy tinh vào tủ rồi khóa cửa.", "Em cất lọ thủy tinh vào tủ rồi khóa cửa."),
    ("Câu sạch", "Con nai đứng im dưới bóng cây.", "Con nai đứng im dưới bóng cây."),
]
reference_keys = set().union(*keys.values())
assert all(text_key(text) not in reference_keys for _, source, target in new_cases for text in (source, target))
engine = SpellInferencePipeline(vit5_path, vit5_meta)
new_results = pd.DataFrame(new_cases, columns=["Nhóm", "Đầu vào", "Tham chiếu"])
new_results["Dự đoán"] = engine.correct_many(new_results["Đầu vào"].tolist())
new_results["Khớp tham chiếu"] = new_results["Dự đoán"] == new_results["Tham chiếu"]
display(new_results)
del engine
gc.collect()
torch.cuda.empty_cache()

## 20. Lưu mô hình và kết quả

Mỗi ZIP chứa checkpoint đã chọn, trọng số, tokenizer, cấu hình và metadata. ZIP báo cáo giữ kết quả từng câu, log, cấu hình, manifest, bảng lỗi và kiểm thử ngoài VSEC để tái hiện thực nghiệm.

In [ ]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for model, checkpoint in selected_models.items():
    target = ARTIFACTS / f"{model}-deploy.zip"
    partial = target.with_suffix(".zip.part")
    with ZipFile(partial, "w", ZIP_DEFLATED) as archive:
        for file in sorted(checkpoint.iterdir()):
            if file.is_file():
                archive.write(file, f"{model}/best_model/{file.name}")
    partial.replace(target)
    print(target)

summary_dir = WORK / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)
for name, table in {"comparison": comparison, "validation-candidates": calibration,
                    "split-summary": split_table, "error-analysis": error_totals.reset_index(),
                    "new-predictions": new_results}.items():
    table.to_csv(summary_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
report_zip = ARTIFACTS / "training-reports.zip"
partial = report_zip.with_suffix(".zip.part")
with ZipFile(partial, "w", ZIP_DEFLATED) as archive:
    archive.write(MANIFEST, "data/splits/split_manifest.json")
    for file in sorted(summary_dir.glob("*.csv")):
        archive.write(file, f"summary/{file.name}")
    for name, path in runs.items():
        for item in ("run_inputs.json", "model_manifest.json", "training_history.json"):
            archive.write(path / item, f"runs/{name}/{item}")
    for file in sorted(RESULTS.glob(f"*/{RUN_ID}-*")):
        archive.write(file, f"results/{file.relative_to(RESULTS).as_posix()}")
partial.replace(report_zip)
print("Tải từ Drive:", report_zip)

## 21. Kết luận thực nghiệm

Đọc kết quả theo cả khả năng phát hiện, sửa đúng và giữ câu sạch. Chênh lệch F1 là mô tả của lần chạy này. Tập VSEC và các câu tự soạn chưa đại diện cho mọi kiểu văn bản tiếng Việt.

In [ ]:
from IPython.display import Markdown

vit5 = reports["ViT5"]["metrics"]
baseline = reports["N-gram + Levenshtein"]["metrics"]
delta = vit5["correction"]["f1"] - baseline["correction"]["f1"]
dominant = error_totals["Vị trí"].idxmax()
f1 = vit5["correction"]["f1"]
detect = vit5["detection"]["f1"]
clean_rate = clean_report["metrics"]["exact_match"]
new_correct = int(new_results["Khớp tham chiếu"].sum())
conclusion = (
    f"- Cấu hình được chọn qua Validation là **{candidate_names[vit5_meta['training_id']]}**.\n"
    f"- Trên VSEC Test, ViT5 đạt Detection F1 **{detect:.2f}** và Correction F1 **{f1:.2f}**. "
    f"Chênh lệch Correction F1 so với baseline là **{delta:+.2f} điểm**.\n"
    f"- Trên Clean Test, ViT5 giữ nguyên **{clean_rate:.2%}** câu. "
    f"Nhóm lỗi có nhiều vị trí nhất trên Test là **{dominant}** ({error_totals.loc[dominant, 'Vị trí']} vị trí).\n"
    f"- ViT5 khớp tham chiếu ở **{new_correct}/{len(new_results)}** câu tự soạn. "
    "Đọc bảng câu sai để xác định các kiểu lỗi cần phân tích tiếp.\n"
)
display(Markdown(conclusion))
(ARTIFACTS / "conclusion.md").write_text(conclusion, encoding="utf-8")